<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Series de Tiempo: la Leche del Centro de Acopio 🥛 (Para Dummies)
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 07
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/07%20-%20Temas%20Avanzados%20en%20Data%20Analysis/Para%20Dummies/00_Series_Tiempo_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 00 del Módulo 07 (Series de tiempo)](../00_Series_de_Tiempo.ipynb). Imagina que trabajas en un **centro de acopio de leche**: cada mes anotas cuántos litros llegaron. Esa lista de números **ordenada en el tiempo** es una **serie de tiempo**, y la pregunta de siempre es: **¿cuánta leche llegará el año que viene?**

| En el acopio | En estadística |
|---|---|
| «Cada año llega más leche porque entran más productores» | **Tendencia** |
| «Todos los junios llega harta leche, todos los diciembres poca» | **Estacionalidad** |
| «Un mes llegó un poco más sin razón clara» | **Ruido** (el azar) |
| «Digo que este mes será igual que el mismo mes del año pasado» | Un **pronóstico ingenuo estacional** |
| «Entre tantos y tantos litros» | Un **intervalo** de pronóstico |

Al terminar podrás explicar:
1. Que una serie se puede **separar en piezas** (lo que sube, lo que se repite y el azar).
2. Por qué **un pronóstico sencillo** (como «igual que el año pasado») es el rival que **todo modelo elegante tiene que vencer**.
3. Por qué los pronósticos se prueban **con meses que el modelo nunca vio**.

> 🔗 Si luego quieres la versión **con fórmulas, pruebas de estacionariedad, ARIMA y validación sin fuga**: [cuaderno estándar](../00_Series_de_Tiempo.ipynb). Taller de todo el módulo: [`../../homeworks/Para Dummies/07_Temas_Avanzados_Hands_On_Dummies.ipynb`](../../homeworks/Para%20Dummies/07_Temas_Avanzados_Hands_On_Dummies.ipynb).

---
## 1. Separar la Serie en Piezas 🧩

Los datos son **inventados**, pero los armamos con tres ingredientes que conocemos: una **subida** de unos 30 mil litros por año, un **vaivén anual** y algo de **azar**. Veamos si, mirando solo los números, podemos **recuperar** esas piezas: el promedio de 12 meses «borra» el vaivén y deja ver lo que sube, y el promedio de cada mes (por ejemplo, de todos los junios) muestra lo que se repite.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Leche (miles de litros) que llega cada mes a un centro de acopio INVENTADO: 8 anios = 96 meses
meses = pd.date_range("2016-01-01", periods=96, freq="MS")
t = np.arange(96)
sube = 400 + 2.5 * t                                           # lo que sube con los anios
ciclo_anual = 50 * np.sin(2 * np.pi * (t % 12 - 2) / 12)       # lo que se repite cada anio
azar = rng.normal(0, 15, 96)                                   # lo que no se puede predecir
serie = pd.Series(sube + ciclo_anual + azar, index=meses)

# Separamos las piezas "a ojo": promedio de 12 meses = lo que sube; promedio de cada mes = lo que se repite
tendencia = serie.rolling(12, center=True).mean()
sin_tend = serie - tendencia
patron = sin_tend.groupby(sin_tend.index.month).mean()
patron = patron - patron.mean()
print("Cuanto sube la leche por anio (aprox.):", round((tendencia.dropna().iloc[-1] - tendencia.dropna().iloc[0]) / ((len(tendencia.dropna()) - 1) / 12), 1), "mil L por anio")
print("Mes con mas leche:", int(patron.idxmax()), "| los dos meses con menos leche:", sorted(int(m) for m in patron.sort_values().index[:2]))
print("Diferencia entre el mejor y el peor mes:", round(float(patron.max() - patron.min()), 1), "mil L")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(serie.index, serie.values, color="#2563eb", label="leche"); ax[0].plot(tendencia.index, tendencia.values, color="#dc2626", lw=2.5, label="lo que sube")
ax[0].legend(); ax[0].set_title("La serie y su tendencia")
ax[1].bar(patron.index, patron.values, color="#16a34a"); ax[1].set_title("Lo que se repite cada anio (por mes)"); ax[1].set_xlabel("mes")
plt.tight_layout(); plt.show()


---
**Lección:** la leche sube **≈ 29.7 mil litros por año** (la subida real que pusimos era 30): el promedio de 12 meses la recuperó casi exacta. Y el vaivén aparece claro: **junio** es el mes con más leche, **diciembre y enero** los de menos, con una diferencia de **≈ 104 mil litros** entre el mejor y el peor mes (la que pusimos era de 100). Separar la serie en piezas es **describir** el pasado; para **pronosticar** hay que seguir esas piezas hacia adelante.

---
## 2. El Pronóstico Más Sencillo es el Rival a Vencer 🥊

Antes de usar cualquier modelo sofisticado, se prueban pronósticos **de una línea**. Guardamos los **últimos 12 meses como «futuro»** (nadie los ve al armar el pronóstico) y comparamos tres ideas: (1) **«igual que el último mes»**, (2) **«igual que el mismo mes del año pasado»**, y (3) **«el mismo mes del año pasado, más lo que suele subir en un año»**. El **error promedio** dice, en miles de litros, cuánto nos equivocamos en promedio.

In [ ]:
# Separamos los ultimos 12 meses como "futuro" (nadie los ve al armar el pronostico)
entrena, prueba = serie.iloc[:84], serie.iloc[84:]
mae = lambda real, pron: float(np.mean(np.abs(np.asarray(real) - np.asarray(pron))))

ingenuo = np.full(12, entrena.iloc[-1])                        # "mañana sera igual que hoy"
mismo_mes = entrena.iloc[-12:].values                          # "este mes sera como el mismo mes del anio pasado"
sube_por_anio = (entrena.iloc[-12:].mean() - entrena.iloc[-24:-12].mean())
mismo_mes_mas_sube = mismo_mes + sube_por_anio                 # ... y ademas sumamos lo que suele subir en un anio

res = pd.Series({"Igual que el ultimo mes": mae(prueba, ingenuo), "Igual que el mismo mes del anio pasado": mae(prueba, mismo_mes),
                 "Mismo mes + lo que sube por anio": mae(prueba, mismo_mes_mas_sube)})
print("Error promedio (en miles de litros) en los 12 meses que NO vimos:\n", res.round(1).to_string())

plt.figure(figsize=(10, 4))
plt.plot(serie.index[-36:], serie.values[-36:], "k", label="real")
plt.plot(prueba.index, ingenuo, color="#dc2626", label="igual que el ultimo mes")
plt.plot(prueba.index, mismo_mes_mas_sube, color="#16a34a", lw=2, label="mismo mes + lo que sube")
plt.legend(); plt.title("Tres pronosticos sencillos para el proximo anio"); plt.show()


---
**Lección:** el error del «igual que el último mes» es **68.3** mil litros: ignora **las dos cosas** que sabemos (que sube y que se repite cada año). «Igual que el mismo mes del año pasado» baja el error a **26.5**: ya capta el vaivén, pero se queda corto porque ignora la subida. Y sumar «lo que sube por año» lo deja en **9.3**. Con tres líneas de código ya tenemos un pronóstico **bueno**; un modelo más elegante (ARIMA, suavizado) solo vale la pena si **le gana** a esto. Dos reglas de oro: (1) **siempre compara con lo sencillo**, y (2) **siempre prueba con datos que el modelo no vio** (si pruebas con los mismos meses con los que armaste el pronóstico, siempre te va a ir bien, y no sirve de nada).

> ⚠️ Estos datos se **inventaron** con una subida y un vaivén muy regulares, justo lo que el pronóstico «mismo mes + sube» captura. En datos reales, con cambios de comportamiento, ganarle al ingenuo es **bastante más difícil**.

---
##### 🛠️ Práctica 1: Promediar dos años

Con `entrena`, `prueba`, `sube_por_anio` y la función `mae` (error promedio) ya definidos:
1. Calcula para cada mes el **promedio de ese mismo mes en los dos últimos años** de `entrena` (`dos_anios`: 12 valores).
2. Como ese promedio está «un año y medio» atrás, suma `sube_por_anio * 1.5` y guarda el pronóstico en `f_prom`.
3. Calcula su error (`error_prom`) y comprueba con `assert` que es **mejor que el de «igual que el último mes»** (`res["Igual que el ultimo mes"]`).

In [ ]:
# Escribe tu código aquí

# dos_anios = ...      # np.mean([entrena.iloc[-12:].values, entrena.iloc[-24:-12].values], axis=0)
# f_prom = ...
# error_prom = mae(prueba, f_prom)

# assert error_prom < res['Igual que el ultimo mes']


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# Promedio de los ultimos 2 anios para cada mes, mas lo que sube por anio
dos_anios = np.mean([entrena.iloc[-12:].values, entrena.iloc[-24:-12].values], axis=0)
f_prom = dos_anios + sube_por_anio * 1.5            # el promedio de dos anios esta "1.5 anios" atras
error_prom = mae(prueba, f_prom)
error_base = res["Mismo mes + lo que sube por anio"]
print(f"Error del promedio de 2 anios = {error_prom:.1f} | error de 'mismo mes + sube' = {error_base:.1f}")
assert error_prom < res["Igual que el ultimo mes"]
```
</details>

---

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿qué diferencia hay entre la **tendencia** y la **estacionalidad** de la leche? ¿Cuál de las dos hace que «igual que el mes pasado» sea un mal pronóstico?
2. Un amigo dice: «mi modelo acierta el 100 % de los meses que usé para armarlo, ¡es perfecto!». ¿Qué le preguntarías antes de creerle?
3. Si el error de tu modelo sofisticado es **8** y el del pronóstico «mismo mes del año pasado + subida» es **9.3**, ¿vale la pena el modelo? ¿Qué más mirarías además del número?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>